# Práctica 4 – Sistemas Inteligentes Aplicados a la Salud  
### Darío Meneses
## Ejercicio 1: Ronda clínica simulada (A* + Algoritmos Genéticos con DEAP)

**Objetivo:** encontrar el mejor orden de visita a las habitaciones indicadas, partiendo del despacho **D20** y terminando en **UCI1**, minimizando el **coste total del recorrido**.

El coste entre dos puntos se calcula como el camino óptimo en el plano del hospital mediante **A\***, considerando las **penalizaciones** del mapa.


In [7]:
# =========================
# IMPORTS 
# =========================

import warnings
warnings.filterwarnings("ignore")

import sys
import types
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from deap import base, creator, tools

# -------------------------
# 1) Importamos el MAPA y ELITISMO desde utils
# -------------------------
from utils.mapa_hospital import mapa_hospital, penalizaciones
from utils.elitism import eaSimpleWithElitism

# -------------------------
# 2) Importamos Nodo y Abiertos desde clases/clases.py
#    
# -------------------------
from clases.clases import Nodo, Abiertos
mod_abiertos = types.ModuleType("clases.abiertos")
mod_abiertos.Abiertos = Abiertos
sys.modules["clases.abiertos"] = mod_abiertos


from utils.busquedaAestrella import busqueda_A_estrella


from clases.clasesVisualizacion import VisualizadorHospital, SolucionesPosiciones

print("Imports OK ")


Imports OK 


In [8]:
# =========================
# DATOS DEL PROBLEMA
# =========================

START = "D20"
END   = "UCI1"

VISITS = ["1S","4S","7S","10S","14S","17S","2N","4N","5N","12N","19N"]

ALL_POINTS = [START] + VISITS + [END]

print("Número total de puntos (incluyendo inicio/fin):", len(ALL_POINTS))
ALL_POINTS


Número total de puntos (incluyendo inicio/fin): 13


['D20',
 '1S',
 '4S',
 '7S',
 '10S',
 '14S',
 '17S',
 '2N',
 '4N',
 '5N',
 '12N',
 '19N',
 'UCI1']

In [9]:
# ============================================================
# CELDA 4 - Localización de los puntos del problema en el mapa
# ============================================================

def find_positions(label, grid):
    """
    Esta función permite localizar todas las posiciones del mapa del hospital
    en las que aparece una determinada etiqueta (habitaciones, despacho, UCI, etc.).

    Args:
        label (str):
            Etiqueta que se quiere localizar en el mapa del hospital
            (por ejemplo 'D20', 'UCI1', '1S', ...).

        grid (list of list):
            Mapa del hospital representado como una matriz de etiquetas.

    Returns:
        list of tuple:
            Lista de coordenadas (fila, columna) donde aparece la etiqueta.
            Si la etiqueta no está presente en el mapa, la lista estará vacía.
    """
    positions = []

    # Recorremos el mapa completo buscando la etiqueta
    for i, row in enumerate(grid):
        for j, cell in enumerate(row):
            if str(cell) == label:
                positions.append((i, j))

    return positions


# ------------------------------------------------------------
# Asociación de cada punto del problema con una posición
# concreta del mapa del hospital
# ------------------------------------------------------------
# En el mapa del hospital algunas zonas (habitaciones, UCI,
# despacho, etc.) ocupan varias celdas. Para simplificar el
# problema de búsqueda, se selecciona una única celda
# representativa para cada punto, concretamente la primera
# que aparece en el recorrido del mapa.
#
# Esta aproximación es suficiente para el objetivo del
# ejercicio, ya que el coste real del desplazamiento se
# calcula posteriormente mediante el algoritmo A*, teniendo
# en cuenta las penalizaciones del mapa.
# ------------------------------------------------------------

POINT_POS = {}

for point in ALL_POINTS:
    pos_list = find_positions(point, mapa_hospital)

    # Comprobación de seguridad: si una etiqueta no existe,
    # el problema no estaría bien definido
    if not pos_list:
        raise ValueError(f"No se encuentra la etiqueta {point} en el mapa.")

    # Se toma la primera aparición como posición representativa
    POINT_POS[point] = pos_list[0]


# Se muestran las posiciones obtenidas, lo cual resulta útil
# tanto para depuración como para la explicación en la memoria
POINT_POS


{'D20': (28, 64),
 '1S': (33, 7),
 '4S': (28, 11),
 '7S': (33, 21),
 '10S': (28, 25),
 '14S': (28, 43),
 '17S': (33, 53),
 '2N': (6, 7),
 '4N': (6, 11),
 '5N': (1, 15),
 '12N': (6, 39),
 '19N': (1, 57),
 'UCI1': (9, 38)}